# From measurements to predictions
**Quantum Machine Learning · Week 3 (Variational quantum classifiers) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qml/w3d2.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**From measurements to predictions.** Yesterday's classifier only used the sign of ⟨Z⟩. Today: turn
measurements into probabilities, train with cross-entropy, read four classes from two qubits, and see what a
finite number of shots does to the predictions.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, ClassicalRegister
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

def classifier(layers=2):
    """Yesterday's circuit: encode once, then trainable rotations and a CNOT in every layer."""
    x, w = ParameterVector("x", 2), ParameterVector("w", 4 * layers)
    qc = QuantumCircuit(2)
    qc.ry(x[0], 0)
    qc.ry(x[1], 1)
    for l in range(layers):
        qc.ry(w[4 * l], 0)
        qc.rz(w[4 * l + 1], 0)
        qc.ry(w[4 * l + 2], 1)
        qc.rz(w[4 * l + 3], 1)
        qc.cx(0, 1)
    return qc

estimator = StatevectorEstimator()

def run(circuit, observables, X, weight_sets):
    """Expectation values of several observables: shape (observables, weight sets, points)."""
    rows = np.array([np.concatenate([W, x]) for W in weight_sets for x in X])
    obs = np.empty((len(observables), 1), dtype=object)             # a column of observables, broadcast over the rows
    for i, o in enumerate(observables):
        obs[i, 0] = o
    evs = estimator.run([(circuit, obs, rows)]).result()[0].data.evs
    return evs.reshape(len(observables), len(weight_sets), len(X))

def values_and_jacobian(circuit, observables, X, W):
    """Values at W, and their derivatives for every weight by the parameter-shift rule, in one batch."""
    sets = [W]
    for k in range(len(W)):
        for shift in (np.pi / 2, -np.pi / 2):
            Wk = W.copy()
            Wk[k] += shift
            sets.append(Wk)
    V = run(circuit, observables, X, sets)
    return V[:, 0], (V[:, 1::2] - V[:, 2::2]) / 2                  # (observables, points), (observables, weights, points)

def xor_data(n=120, seed=0):
    rng = np.random.default_rng(seed)
    corner = rng.choice([-1, 1], size=(n, 2))
    return corner + 0.35 * rng.normal(size=(n, 2)), (corner[:, 0] * corner[:, 1] < 0).astype(int)

X, y = xor_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=40, random_state=0, stratify=y)
scaler = MinMaxScaler((-np.pi / 2, np.pi / 2)).fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
circuit, Z1 = classifier(), SparsePauliOp("ZI")                     # ⟨Z⟩ of qubit 1, as yesterday
print("ready:", len(X_train), "training points,", len(X_test), "test points")

## A probability straight from the Born rule
$p(\text{class }1) = P(\text{qubit 1 reads }1) = \frac{1 - \langle Z\rangle}{2}$. Train it with cross-entropy,
the loss of logistic regression.

In [ ]:
def cross_entropy(p, y):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def train_born(epochs=60, lr=0.3, seed=0):
    W = 0.6 * np.random.default_rng(seed).normal(size=8)
    for epoch in range(epochs):
        f, J = values_and_jacobian(circuit, [Z1], X_train, W)
        p = np.clip((1 - f[0]) / 2, 1e-9, 1 - 1e-9)
        dL_df = -(p - y_train) / (2 * p * (1 - p) * len(p))          # chain rule through p = (1 - f) / 2
        W = W - lr * (J[0] @ dL_df)
        if epoch % 20 == 0:
            print(f"epoch {epoch:2d}: cross-entropy {cross_entropy(p, y_train):.3f}")
    return W

W_born = train_born()
p_born = (1 - run(circuit, [Z1], X_test, [W_born])[0, 0]) / 2
print(f"test accuracy {np.mean((p_born > 0.5) == y_test):.3f}, test cross-entropy {cross_entropy(p_born, y_test):.3f}")
print("most confident probabilities:", p_born.min().round(3), "and", p_born.max().round(3))

## A classical scale on top
The circuit's ⟨Z⟩ rarely reaches ±1, so those probabilities stay timid. Add two classical numbers, a scale $a$
and a shift $b$: $p = \sigma(-a\langle Z\rangle + b)$, trained together with the circuit.

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))

def train_scaled(epochs=60, lr=0.3, seed=0):
    W, a, b = 0.6 * np.random.default_rng(seed).normal(size=8), 2.0, 0.0   # a = 2 matches the Born slope at ⟨Z⟩ = 0
    for epoch in range(epochs):
        f, J = values_and_jacobian(circuit, [Z1], X_train, W)
        p = sigmoid(-a * f[0] + b)
        dz = (p - y_train) / len(p)                                  # cross-entropy's gradient for the sigmoid's input
        W, a, b = W - lr * (J[0] @ (-a * dz)), a - lr * np.sum(-f[0] * dz), b - lr * np.sum(dz)
    return W, a, b

W_s, a, b = train_scaled()
f_test = run(circuit, [Z1], X_test, [W_s])[0, 0]
p_scaled = sigmoid(-a * f_test + b)
print(f"learned scale a = {a:.2f}, shift b = {b:.2f}")
print(f"test accuracy {np.mean((p_scaled > 0.5) == y_test):.3f}, test cross-entropy {cross_entropy(p_scaled, y_test):.3f}")

import matplotlib.pyplot as plt
z = np.linspace(-1, 1, 200)
fig, ax = plt.subplots(figsize=(6.2, 3.6), layout="constrained")
ax.plot(z, (1 - z) / 2, color="#8a85a3", lw=2, ls="--", label=r"Born rule: $(1 - \langle Z\rangle)/2$")
ax.plot(z, sigmoid(-a * z + b), color="#7b5cfa", lw=2.4, label=fr"scaled: $\sigma(-{a:.1f}\,\langle Z\rangle {b:+.2f})$")
ax.scatter(f_test, sigmoid(-a * f_test + b), c=np.where(y_test == 0, "#ff8a50", "#5577f5"), s=26, zorder=3, edgecolors="white", linewidths=0.6)
ax.set_xlabel(r"$\langle Z\rangle$ of qubit 1")
ax.set_ylabel("probability of class 1")
ax.legend(frameon=False, loc="upper right")
plt.show()

## Shots
Hardware never hands you ⟨Z⟩, only shots. With $S$ shots the estimate wobbles by about
$\sqrt{(1 - \langle Z\rangle^2)/S}$, so points near the boundary can land on the wrong side.

In [ ]:
measured = circuit.copy()
measured.add_register(ClassicalRegister(1, "c"))
measured.measure(1, 0)                                              # only qubit 1 is read
sampler = StatevectorSampler(seed=np.random.default_rng(1))
rows = np.array([np.concatenate([W_s, x]) for x in X_test])
exact = np.where(f_test < 0, 1, 0)
for shots in (10, 100, 1000):
    accs, flips = [], []
    for repeat in range(20):
        ones = sampler.run([(measured, rows)], shots=shots).result()[0].data.c.bitcount().sum(axis=-1)
        estimate = 1 - 2 * ones / shots                             # ⟨Z⟩ estimated from the counts
        predicted = np.where(estimate < 0, 1, np.where(estimate > 0, 0, np.random.default_rng(repeat).integers(0, 2, len(estimate))))
        accs.append(np.mean(predicted == y_test))
        flips.append(np.sum(predicted != exact))
    print(f"{shots:5d} shots: accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f}, predictions that differ from the exact ones: {np.mean(flips):.1f} of {len(y_test)}")
print(f"exact ⟨Z⟩:  accuracy {np.mean(exact == y_test):.3f}")

## Four classes from two qubits
Measure both qubits and the Born rule gives four probabilities, one per bitstring, that already add up to 1:
a four-class classifier with no softmax. Each probability is the expectation value of a projector,
so the parameter-shift rule still gives its gradient.

In [ ]:
def projector(b0, b1):
    s0, s1 = (-1) ** b0, (-1) ** b1                                 # |b⟩⟨b| = (I ± Z)/2 on each qubit
    return SparsePauliOp.from_list([("II", 0.25), ("IZ", 0.25 * s0), ("ZI", 0.25 * s1), ("ZZ", 0.25 * s0 * s1)])

projectors = [projector(k // 2, k % 2) for k in range(4)]          # class k ↔ qubit 0 reads k // 2, qubit 1 reads k % 2
rng = np.random.default_rng(0)
sign = rng.choice([-1, 1], size=(160, 2))
X4 = sign + 0.35 * rng.normal(size=(160, 2))
y4 = 2 * (sign[:, 0] > 0) + (sign[:, 1] > 0)                       # four quadrants, four classes
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=60, random_state=0, stratify=y4)
scaler4 = MinMaxScaler((-np.pi / 2, np.pi / 2)).fit(X4_train)
X4_train, X4_test = scaler4.transform(X4_train), scaler4.transform(X4_test)

W4 = 0.6 * np.random.default_rng(0).normal(size=8)
idx = np.arange(len(y4_train))
for epoch in range(60):
    P, J = values_and_jacobian(circuit, projectors, X4_train, W4)  # P: (4 classes, points)
    p_true = np.clip(P[y4_train, idx], 1e-9, 1)
    W4 = W4 - 0.5 * (-(J[y4_train, :, idx] / p_true[:, None]).mean(axis=0))   # gradient of -mean log p(true class)
    if epoch % 20 == 0:
        print(f"epoch {epoch:2d}: cross-entropy {-np.mean(np.log(p_true)):.3f}")
P_test = run(circuit, projectors, X4_test, [W4])[:, 0]
print(f"test accuracy {np.mean(P_test.argmax(axis=0) == y4_test):.3f}")
print("first test point: probabilities", P_test[:, 0].round(3), "sum", P_test[:, 0].sum().round(3), "true class", y4_test[0])

## Your turn: which points flip?
Measure the trained XOR classifier ($p = \sigma(-a\langle Z\rangle + b)$ aside, just the sign of the estimate)
with 20 shots per point, 200 times over. For each test point, how often does the prediction differ from the
exact one? Plot that flip rate against $|\langle Z\rangle|$.

In [ ]:
shots, repeats = 20, 200
p1 = (1 - f_test) / 2                                               # chance that qubit 1 reads 1, per test point
rng = np.random.default_rng(0)
# TODO: draw rng.binomial(shots, p1, size=(repeats, len(p1))) counts of ones, estimate ⟨Z⟩ = 1 - 2·ones/shots,
#       compare its sign with the exact one, and plot each point's flip rate against np.abs(f_test)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
shots, repeats = 20, 200
p1 = (1 - f_test) / 2
rng = np.random.default_rng(0)
ones = rng.binomial(shots, p1, size=(repeats, len(p1)))
estimate = 1 - 2 * ones / shots
tie = rng.integers(0, 2, size=estimate.shape)                       # a 10-10 split is a coin toss
predicted = np.where(estimate < 0, 1, np.where(estimate > 0, 0, tie))
flip_rate = np.mean(predicted != exact, axis=0)

from scipy.stats import norm
zz = np.linspace(0.001, 1, 200)
fig, ax = plt.subplots(figsize=(6.2, 3.6), layout="constrained")
ax.scatter(np.abs(f_test), flip_rate, color="#7b5cfa", s=30, zorder=3, label="test points")
ax.plot(zz, norm.cdf(-zz * np.sqrt(shots) / np.sqrt(1 - zz ** 2 + 1e-12)), color="#ff8a50", lw=2, label="normal approximation")
ax.set_xlabel(r"$|\langle Z\rangle|$: distance from the boundary, in score")
ax.set_ylabel("how often the prediction flips")
ax.set_title("20 shots per prediction")
ax.legend(frameon=False)
plt.show()
print(f"points that flip more than 10% of the time: {np.sum(flip_rate > 0.1)} of {len(flip_rate)}")
print("They are exactly the points with small |⟨Z⟩|: shots only matter near the boundary.")